# NPU

在下面的表格中，我们回顾了过去20年计算能力的快速增长。简而言之，自2000年以来，GPU/NPU性能每十年增长1000倍。

本节，我们将讨论如何利用这种计算性能进行研究。首先是如何使用单个NPU，然后是如何使用多个NPU和多个服务器（具有多个NPU）。

我们先看看如何使用单个 Ascend NPU 进行计算。首先，确保至少安装了一个 Ascend NPU。然后，下载 [CANN 工具链](https://www.hiascend.com/zh/software/cann) 并按照提示设置适当的路径。当这些准备工作完成，就可以使用`npu-smi`命令来**查看 NPU 信息。**

| 年代 | 数据规模              | 内存   | 每秒浮点运算        |
| :--- | :--- | :--- | :--- |
| 1970 | 100 （鸢尾花卉）      | 1 KB   | 100 KF (Intel 8080) |
| 1980 | 1 K （波士顿房价）    | 100 KB | 1 MF (Intel 80186)  |
| 1990 | 10 K （光学字符识别） | 10 MB  | 10 MF (Intel 80486) |
| 2000 | 10 M （网页）         | 100 MB | 1 GF (Intel Core)   |
| 2010 | 10 G （广告）         | 1 GB   | 1 TF (Nvidia C2050) |
| 2020 | 1 T （社交网络）      | 100 GB | 1 PF (Nvidia DGX-2) |

---

## 环境准备

In [ ]:
%pip install pypto==0.2.0 torch torch_npu numpy

In [ ]:
import os
import sys
os.environ["TILE_FWK_DEVICE_ID"] = "0"

# 本 notebook 所在目录的父目录中有 src 包
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import torch
import torch_npu
from torch import nn

from src.pto_layers import PyPTOLinear, PyPTOReLU

---

In [3]:
!npu-smi info

+------------------------------------------------------------------------------------------------+
| npu-smi 25.5.5                   Version: 25.5.5                                               |
+---------------------------+---------------+----------------------------------------------------+
| NPU   Name                | Health        | Power(W)    Temp(C)           Hugepages-Usage(page)|
| Chip  Phy-ID              | Bus-Id        | AICore(%)   Memory-Usage(MB)  HBM-Usage(MB)        |
+===========================+===============+====================================================+
| 3     Ascend910           | OK            | 164.4       43                0    / 0             |
| 0     6                   | 0000:0B:00.0  | 0           0    / 0          4777 / 65536         |
+------------------------------------------------------------------------------------------------+
| 3     Ascend910           | OK            | -           42                0    / 0             |
| 1     7 

在 PyTorch 中，每个数组都有一个设备（device），我们通常将其称为环境（context）。默认情况下，所有变量和相关的计算都分配给 CPU。有时环境可能是 NPU。当我们跨多个服务器部署作业时，事情会变得更加棘手。通过智能地将数组分配给环境，我们可以最大限度地减少在设备之间传输数据的时间。例如，当在带有 NPU 的服务器上训练神经网络时，我们通常希望模型的参数在 NPU 上。

要运行此部分中的程序，至少需要两个 NPU。注意，对大多数桌面计算机来说，这可能是奢侈的，但在云中很容易获得。本书的其他章节大都不需要多个 NPU，而本节只是为了展示数据如何在不同的设备之间传递。

---

## **计算设备**

我们可以指定用于存储和计算的设备，如 CPU 和 NPU。默认情况下，张量是在内存中创建的，然后使用 CPU 计算它。

在 PyTorch 中，CPU 和 NPU 可以用 `torch.device('cpu')` 和 `torch.device('npu')` 表示。应该注意的是，`cpu` 设备意味着所有物理 CPU 和内存，这意味着 PyTorch 的计算将尝试使用所有 CPU 核心。然而，`npu` 设备只代表一张卡和相应的显存。如果有多个 NPU，我们使用 `torch.device(f'npu:{i}')` 来表示第 $i$ 块 NPU（$i$ 从 0 开始）。另外，`npu:0` 和 `npu` 是等价的。

In [4]:
torch.device('cpu'), torch.device('npu'), torch.device('npu:1')

(device(type='cpu'), device(type='npu'), device(type='npu', index=1))

我们可以**查询可用 NPU 的数量。**

In [5]:
torch.npu.device_count()

2

现在我们定义了两个方便的函数，**这两个函数允许我们在不存在所需所有 NPU 的情况下运行代码。**

In [6]:
def try_npu(i=0):  #@save
    """如果存在，则返回 npu(i)，否则返回 cpu()"""
    if torch.npu.device_count() >= i + 1:
        return torch.device(f'npu:{i}')
    return torch.device('cpu')

def try_all_npus():  #@save
    """返回所有可用的 NPU，如果没有 NPU，则返回 [cpu(),]"""
    devices = [torch.device(f'npu:{i}')
             for i in range(torch.npu.device_count())]
    return devices if devices else [torch.device('cpu')]

try_npu(), try_npu(10), try_all_npus()

(device(type='npu', index=0),
 device(type='cpu'),
 [device(type='npu', index=0), device(type='npu', index=1)])

---

## 张量与 NPU

我们可以**查询张量所在的设备。** 默认情况下，张量是在 CPU 上创建的。

In [7]:
x = torch.tensor([1, 2, 3])
x.device

device(type='cpu')

需要注意的是，无论何时我们要对多个项进行操作，它们都必须在同一个设备上。例如，如果我们对两个张量求和，我们需要确保两个张量都位于同一个设备上，否则框架将不知道在哪里存储结果，甚至不知道在哪里执行计算。

---

### **存储在 NPU 上**

有几种方法可以在 NPU 上存储张量。例如，我们可以在创建张量时指定存储设备。接下来，我们在第一个 `npu` 上创建张量变量 `X`。在 NPU 上创建的张量只消耗这个 NPU 的显存。我们可以使用 `npu-smi` 命令查看显存使用情况。一般来说，我们需要确保不创建超过 NPU 显存限制的数据。

In [9]:
X = torch.ones(2, 3, device=try_npu())
X

tensor([[1., 1., 1.],
        [1., 1., 1.]], device='npu:0')

假设我们至少有两个 NPU，下面的代码将在**第二个 NPU 上创建一个随机张量。**

In [10]:
Y = torch.rand(2, 3, device=try_npu(1))
Y

tensor([[0.5948, 0.6832, 0.2111],
        [0.8515, 0.2021, 0.0683]], device='npu:1')

---

### 复制

如果我们**要计算 `X + Y`，我们需要决定在哪里执行这个操作**。例如，我们可以将 `X` 传输到第二个 NPU 并在那里执行操作。*不要*简单地 `X` 加上 `Y`，因为这会导致异常，运行时引擎不知道该怎么做：它在同一设备上找不到数据会导致失败。由于 `Y` 位于第二个 NPU 上，所以我们需要将 `X` 移到那里，然后才能执行相加运算。

In [11]:
Z = X.npu(1)
print(X)
print(Z)

tensor([[1., 1., 1.],
        [1., 1., 1.]], device='npu:0')
tensor([[1., 1., 1.],
        [1., 1., 1.]], device='npu:1')


**现在数据在同一个 NPU 上（`Z` 和 `Y` 都在），我们可以将它们相加。**

In [12]:
Y + Z

tensor([[1.5948, 1.6832, 1.2111],
        [1.8515, 1.2021, 1.0683]], device='npu:1')

假设变量 `Z` 已经存在于第二个 NPU 上。如果我们还是调用 `Z.npu(1)` 会发生什么？它将返回 `Z`，而不会复制并分配新内存。

In [13]:
Z.npu(1) is Z

True

---

### 旁注

人们使用 NPU 来进行机器学习，因为单个 NPU 相对运行速度快。但是在设备（CPU、NPU 和其他机器）之间传输数据比计算慢得多。这也使得并行化变得更加困难，因为我们必须等待数据被发送（或者接收），然后才能继续进行更多的操作。这就是为什么拷贝操作要格外小心。根据经验，多个小操作比一个大操作糟糕得多。此外，一次执行几个操作比代码中散布的许多单个操作要好得多。如果一个设备必须等待另一个设备才能执行其他操作，那么这样的操作可能会阻塞。这有点像排队订购咖啡，而不像通过电话预先订购：当客人到店的时候，咖啡已经准备好了。

最后，当我们打印张量或将张量转换为 NumPy 格式时，如果数据不在内存中，框架会首先将其复制到内存中，这会导致额外的传输开销。更糟糕的是，它现在受制于全局解释器锁，使得一切都得等待 Python 完成。

---

## **神经网络与 NPU**

类似地，神经网络模型可以指定设备。下面的代码将模型参数放在 NPU 上。这里我们使用本章前面定义的 `PyPTOLinear`，它与 `nn.Linear` 用法一致，但底层 kernel 由 PyPTO 实现。

In [14]:
net = nn.Sequential(PyPTOLinear(3, 1))
net = net.to(device=try_npu())

在接下来的几章中，我们将看到更多关于如何在 NPU 上运行模型的例子，因为它们将变得更加计算密集。

当输入为 NPU 上的张量时，模型将在同一 NPU 上计算结果。

In [15]:
net(X)

tensor([[1.4827],
        [1.4827]], device='npu:0', grad_fn=<ViewBackward0>)

让我们**确认模型参数存储在同一个 NPU 上。**

In [16]:
net[0].weight.data.device

device(type='npu', index=0)

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <p style="margin: 0 0 10px 0;">原文使用 NVIDIA GPU + CUDA 实现相同演示。核心 API 对照如下：</p>
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0; font-family: Consolas, monospace;">
torch.device('cpu'), torch.device('cuda'), torch.device('cuda:1')
torch.cuda.device_count()

def try_gpu(i=0):
    if torch.cuda.device_count() >= i + 1:
        return torch.device(f'cuda:{i}')
    return torch.device('cpu')

X = torch.ones(2, 3, device=try_gpu())   # device='cuda:0'
Y = torch.rand(2, 3, device=try_gpu(1))  # device='cuda:1'
Z = X.cuda(1)                             # 跨卡拷贝
Y + Z                                     # 同卡相加
Z.cuda(1) is Z                            # 幂等

net = nn.Sequential(nn.Linear(3, 1))
net = net.to(device=try_gpu())
net(X)
net[0].weight.data.device                 # device(type='cuda', index=0)</pre>
    <p style="margin: 10px 0 0 0;">本节将上述 <code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">cuda</code> 全部替换为 <code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">npu</code>，并使用 <code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">import torch_npu</code> 注册 NPU 后端，<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">nn.Linear</code> 替换为 <code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">PyPTOLinear</code>。</p>
  </div>
</details>

总之，只要所有的数据和参数都在同一个设备上，我们就可以有效地学习模型。在下面的章节中，我们将看到几个这样的例子。

---

## 小结

* 我们可以指定用于存储和计算的设备，例如 CPU 或 NPU。默认情况下，数据在主内存中创建，然后使用 CPU 进行计算。
* 深度学习框架要求计算的所有输入数据都在同一设备上，无论是 CPU 还是 NPU。
* 不经意地移动数据可能会显著降低性能。一个典型的错误如下：计算 NPU 上每个小批量的损失，并在命令行中将其报告给用户（或将其记录在 NumPy `ndarray` 中）时，将触发全局解释器锁，从而使所有 NPU 阻塞。最好是为 NPU 内部的日志分配内存，并且只移动较大的日志。

---
## 练习

1. 尝试一个计算量更大的任务，比如大矩阵的乘法，看看 CPU 和 NPU 之间的速度差异。再试一个计算量很小的任务呢？
1. 我们应该如何在 NPU 上读写模型参数？
1. 测量计算1000个 $100 \times 100$ 矩阵的矩阵乘法所需的时间，并记录输出矩阵的 Frobenius 范数，一次记录一个结果，而不是在 NPU 上保存日志并仅传输最终结果。
1. 测量同时在两个 NPU 上执行两个矩阵乘法与在一个 NPU 上按顺序执行两个矩阵乘法所需的时间。提示：应该看到近乎线性的缩放。

详细参考答案见[05.06_reference_answer](./answers/05.06_reference_answer.ipynb)

#### **参考答案(PyPTO)**

In [ ]:
!cat ./answers/txt/05.06_reference_pypto.txt

#### **参考答案(PyTorch)**

In [ ]:
!cat ./answers/txt/05.06_reference_pytorch.txt